# 전처리 · 13단어 전문가 자료와 팀 영상 325개

In [ ]:
import os
import sys
from pathlib import Path

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "modules/vocabulary.py").is_file())
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
os.environ["SIGNBRIDGE_VOCABULARY"] = "assistant_demo"
from modules.vocabulary import NUM_CLASSES
if NUM_CLASSES != 13:
    raise RuntimeError("Restart the kernel to load the assistant_demo vocabulary.")
import csv
import json
from collections import Counter, defaultdict

import numpy as np
from modules.vocabulary import LABEL_MAP, WORDS
from modules.preprocessing import VIDEO_EXTENSIONS, load_manifest, parse_team_filename, run_preprocessing
from scripts.data.migrate_expert_npy import migrate

source = Path("data/expert_assistant_demo")
videos = Path.home() / "Downloads" / "영상_데이터"
root = Path("data/processed_assistant_demo_325").resolve()
overwrite = False  # 같은 이름의 원본 영상을 교체했다면 True

## 입력 확인 · 시연자, 촬영 방향, 단어

In [ ]:
clips = sorted(p for p in videos.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
entries = [parse_team_filename(p) for p in clips]
counts = Counter(word for signer, angle, word in entries)
if len(set(entries)) != 325 or len(entries) != 325 or any(counts[key] != 25 for key in LABEL_MAP):
    raise ValueError(f"Expected 325 distinct signer/angle/word recordings, 25 per class: {counts}")
print({WORDS[key]: counts[key] for key in LABEL_MAP})

## 전문가 NPY 변환

In [ ]:
# Fail before creating output files if the source arrays cannot cover the vocabulary.
source_map_path = source / "core_label_map.json"
source_map = json.loads(source_map_path.read_text(encoding="utf-8"))
missing = [f"{key} ({word})" for key, word in WORDS.items() if key not in source_map]
if missing:
    raise ValueError("Expert NPY source is missing active classes: " + ", ".join(missing)
                     + ". Supply arrays and a matching core_label_map.json containing all requested classes.")
root.mkdir(parents=True, exist_ok=True)
map_path = root / "core_label_map.json"
if map_path.exists() and json.loads(map_path.read_text(encoding="utf-8")) != LABEL_MAP:
    raise ValueError("Output vocabulary differs. Use a new --processed-root.")
(root / "core_label_map.json").write_text(json.dumps(LABEL_MAP, indent=2), encoding="utf-8")
(root / "core_ksl_word_dictionary.json").write_text(
    json.dumps(WORDS, ensure_ascii=False, indent=2), encoding="utf-8")
manifest = root / "manifest.csv"
if overwrite or not any(row["domain"] == "expert" for row in load_manifest(manifest)):
    count, _ = migrate(source / "X_train.npy", source / "y_train.npy",
                       root, overwrite=overwrite)
    print(f"Selected {count} expert sequences", flush=True)

## 팀 영상에서 관절 추출 · 기존 시퀀스는 재사용

In [ ]:
processed, failed = run_preprocessing(project, root / "raw_videos", root, ("team",), overwrite, videos.resolve())
if failed:
    raise RuntimeError(f"{failed} clips failed; inspect preprocessing_failures.txt")
print(f"{processed} team clips")

## 시퀀스 크기·손 검출률·클래스별 수량 검사

In [ ]:
with (root / "manifest.csv").open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
counts = Counter((row["domain"], row["class_key"]) for row in rows)
signer_counts = Counter(row["signer_id"] for row in rows if row["domain"] == "team")
coverage = defaultdict(list)
low_coverage = []
for row in rows:
    with np.load(root / row["path"]) as archive:
        sequence = archive["x"]
    if sequence.shape != (30, 452) or not np.isfinite(sequence).all():
        raise ValueError(f"Invalid sequence {row['path']}")
    if row["domain"] == "team":
        ratio = float(np.mean(np.any(sequence[:, -2:] > 0.5, axis=1)))
        coverage[row["signer_id"]].append(ratio)
        if ratio < 0.5:
            low_coverage.append({"source_video": row["source_video"],
                                 "word": WORDS[row["class_key"]],
                                 "resampled_any_hand_fraction": ratio})
report = {
    "samples": len(rows), "shape": [30, 452], "all_finite": True,
    "class_counts": {WORDS[key]: {domain: counts[domain, key] for domain in ("expert", "team")}
                     for key in LABEL_MAP},
    "signer_counts": dict(signer_counts),
    "resampled_any_hand_fraction_by_signer": {
        signer: float(np.mean(ratios)) for signer, ratios in coverage.items()},
    "low_coverage_clips": low_coverage,
    "coverage_note": "This measures resampled masks over the entire clip, including preparation/rest frames; it is not detector accuracy.",
}
(root / "dataset_summary.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(report["class_counts"])